# 5G‑NIDD — CTGAN Augmentation Training

This notebook trains **CTGAN** for **tabular data augmentation** on 5G‑NIDD.

It supports **both modes**:
- **Unconditional**: learns `p(X)` (no label column required).
- **Conditional**: learns `p(X, y)` and enables **label‑controlled generation** (e.g., 10,000 rows per attack type).

Inputs expected in `OUT_DIR`:
- `train_input.csv`
- `train_input_with_label.csv`
- `prep_metadata_aug.json`


## 0) Setup

In [ ]:
import os, json, time, platform
import numpy as np
import pandas as pd
import time
import pkg_resources
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from ctgan import CTGAN
import torch
pd.set_option("display.max_columns", 200)

import re

def safe_filename(name: str) -> str:
    name = str(name).strip()
    name = re.sub(r"[^\w\-\.]+", "_", name)
    name = re.sub(r"_+", "_", name).strip("_")
    return name or "unknown"


## 1) Configuration

In [ ]:
OUT_DIR = "5G-NIDD_Aug_CTGAN/UDPScan"
META_AUG = os.path.join(OUT_DIR, "prep_metadata_aug.json")

TRAIN_UNCOND = os.path.join(OUT_DIR, "train_input.csv")
TRAIN_COND   = os.path.join(OUT_DIR, "train_input_with_label.csv")

TRAIN_MODE = "unconditional"  # "unconditional" | "conditional" | "both"

GENERATE_PER_CLASS = True
N_PER_CLASS = 80000

TOTAL_SYN_BALANCED = 160000  # only used if GENERATE_PER_CLASS = False
BENIGN_NAME = "Benign"

UNCOND_SAMPLES = 10000 # only used if TRAIN_MODE = "unconditional" or "both"

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

print("OUT_DIR:", OUT_DIR)
print("TRAIN_MODE:", TRAIN_MODE)


## 2) Load metadata + training matrices

In [ ]:
assert os.path.exists(META_AUG), f"Missing metadata: {META_AUG}"
aug = json.load(open(META_AUG))

read_csv = lambda p: pd.read_csv(p, engine="python", on_bad_lines="skip")

Xtr_uncond = read_csv(TRAIN_UNCOND) if os.path.exists(TRAIN_UNCOND) else None
Xtr_cond   = read_csv(TRAIN_COND)   if os.path.exists(TRAIN_COND)   else None

print("train_input.csv:", None if Xtr_uncond is None else Xtr_uncond.shape)
print("train_input_with_label.csv:", None if Xtr_cond is None else Xtr_cond.shape)

cfg_uncond = aug.get("augmentation_unconditional", {})
cfg_cond   = aug.get("augmentation_conditional", {})
LABEL_COL = cfg_cond.get("label_col", "Attack Type")

if TRAIN_MODE in ("unconditional", "both"):
    assert Xtr_uncond is not None, "Missing train_input.csv"
if TRAIN_MODE in ("conditional", "both"):
    assert Xtr_cond is not None, "Missing train_input_with_label.csv"
    assert LABEL_COL in Xtr_cond.columns, f"Label col '{LABEL_COL}' missing"

print("LABEL_COL:", LABEL_COL)


## 3) Input Sanity Check

In [ ]:
def _is_binary_series(s: pd.Series) -> bool:
    s = pd.to_numeric(s, errors="coerce").dropna()
    if len(s) == 0:
        return False
    u = set(s.unique().tolist())
    return u.issubset({0, 1, 0.0, 1.0})

def sanity_check_input_ctgan():
    # Basic existence
    assert TRAIN_MODE in ("unconditional", "conditional", "both")
    assert os.path.exists(META_AUG), f"Missing {META_AUG}"

    # Unconditional checks
    if TRAIN_MODE in ("unconditional", "both"):
        assert Xtr_uncond is not None and len(Xtr_uncond) > 0, "Xtr_uncond empty"
        # No label col expected in unconditional
        if "Attack Type" in Xtr_uncond.columns and "Attack Type" == LABEL_COL:
            print("Note: LABEL_COL appears in unconditional data; usually it should NOT.")
        # NaNs
        nan_rate = Xtr_uncond.isna().mean().mean()
        assert nan_rate < 0.01, f"Too many NaNs in Xtr_uncond (avg rate={nan_rate:.4f}). Fix prep/imputation."
        # Discrete columns should be binary
        disc = cfg_uncond.get("discrete_columns", [])
        missing = [c for c in disc if c not in Xtr_uncond.columns]
        assert not missing, f"Missing discrete columns in Xtr_uncond: {missing[:10]}"
        bad = [c for c in disc if not _is_binary_series(Xtr_uncond[c])]
        assert not bad, f"These discrete columns are not strictly binary in Xtr_uncond: {bad[:10]}"

    # Conditional checks
    if TRAIN_MODE in ("conditional", "both"):
        assert Xtr_cond is not None and len(Xtr_cond) > 0, "Xtr_cond empty"
        assert LABEL_COL in Xtr_cond.columns, f"Missing LABEL_COL={LABEL_COL} in Xtr_cond"
        # NaNs
        nan_rate = Xtr_cond.isna().mean().mean()
        assert nan_rate < 0.01, f"Too many NaNs in Xtr_cond (avg rate={nan_rate:.4f}). Fix prep/imputation."
        # Label inventory
        classes = sorted(Xtr_cond[LABEL_COL].dropna().astype(str).unique().tolist())
        assert len(classes) >= 2, "Need >=2 classes for conditional augmentation."
        counts = Xtr_cond[LABEL_COL].astype(str).value_counts()
        assert counts.min() > 10, f"Some classes too small for learning. Min count={counts.min()}"
        # Discrete columns (incl. label) should be present and binary (label can be non-binary categorical)
        disc = cfg_cond.get("discrete_columns", [])
        missing = [c for c in disc if c not in Xtr_cond.columns]
        assert not missing, f"Missing discrete columns in Xtr_cond: {missing[:10]}"
        # Check binary one-hots only (exclude LABEL_COL)
        disc_bin = [c for c in disc if c != LABEL_COL]
        bad = [c for c in disc_bin if not _is_binary_series(Xtr_cond[c])]
        assert not bad, f"These discrete (one-hot) columns are not strictly binary in Xtr_cond: {bad[:10]}"

    print("Validation OK")

sanity_check_input_ctgan()


## 4) Train CTGAN models

In [ ]:
train_start_time = time.time()

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE)

ctgan_uncond = None
ctgan_cond   = None

discrete_uncond = cfg_uncond.get("discrete_columns", [])
discrete_cond   = cfg_cond.get("discrete_columns", [])

EPOCHS = 500
BATCH  = 500
PAC    = 10

if TRAIN_MODE in ("unconditional", "both"):
    ctgan_uncond = CTGAN(epochs=EPOCHS, batch_size=BATCH, cuda=(DEVICE=="cuda"), verbose=True, pac=PAC)
    print("Fitting UNCONDITIONAL CTGAN ...")
    t0 = time.time()
    ctgan_uncond.fit(Xtr_uncond, discrete_columns=discrete_uncond)
    print("Done. minutes:", round((time.time()-t0)/60, 2))

if TRAIN_MODE in ("conditional", "both"):
    if LABEL_COL not in discrete_cond:
        discrete_cond = sorted(set(discrete_cond + [LABEL_COL]))
    ctgan_cond = CTGAN(epochs=EPOCHS, batch_size=BATCH, cuda=(DEVICE=="cuda"), verbose=True, pac=PAC)
    print("Fitting CONDITIONAL CTGAN ...")
    t0 = time.time()
    ctgan_cond.fit(Xtr_cond, discrete_columns=discrete_cond)
    print("Done. minutes:", round((time.time()-t0)/60, 2))

train_end_time = time.time()
train_duration_sec = train_end_time - train_start_time


## 5) Sampling 
### CTGAN uses oversample and then filter to get exact per‑class counts

In [ ]:
UNCOND_OUT = os.path.join(OUT_DIR, "synthetic_ctgan_unconditional.csv")
if ctgan_uncond is not None:
    syn_u = ctgan_uncond.sample(UNCOND_SAMPLES)
    syn_u.to_csv(UNCOND_OUT, index=False)
    print("Wrote:", UNCOND_OUT, syn_u.shape)


In [ ]:
if ctgan_cond is not None:
    classes = sorted(Xtr_cond[LABEL_COL].dropna().astype(str).unique().tolist())
    print("Classes:", classes)

    def sample_exact_class_ctgan(label_value: str, n_needed: int, oversample_factor: int = 6, min_draw: int = 20000):
        remaining = n_needed
        parts = []
        while remaining > 0:
            draw = max(min_draw, remaining * oversample_factor)
            tmp = ctgan_cond.sample(draw)
            tmp = tmp[tmp[LABEL_COL].astype(str) == str(label_value)]
            take = tmp.head(remaining)
            parts.append(take)
            remaining -= len(take)
            if len(take) == 0:
                min_draw = int(min_draw * 1.5)
        return pd.concat(parts, axis=0).reset_index(drop=True)

    if GENERATE_PER_CLASS:
        mal_labels = [c for c in classes if c != str(BENIGN_NAME)]
        assert len(mal_labels) > 0, "No malicious classes found"

        PER_CLASS_DIR = os.path.join(OUT_DIR, "synthetic_per_class")
        os.makedirs(PER_CLASS_DIR, exist_ok=True)

        parts_all = []

        # 1) Benign = sum of all malicious
        n_benign = len(mal_labels) * N_PER_CLASS
        if str(BENIGN_NAME) in classes:
            df_ben = sample_exact_class_ctgan(str(BENIGN_NAME), n_benign)
            parts_all.append(df_ben)

            out_ben = os.path.join(
                PER_CLASS_DIR,
                f"synthetic_{safe_filename(BENIGN_NAME)}_{n_benign}.csv"
            )
            df_ben.to_csv(out_ben, index=False)
            print(f"Saved {BENIGN_NAME}: {out_ben}")

        # 2) Each malicious class -> N_PER_CLASS
        for lab in mal_labels:
            df_lab = sample_exact_class_ctgan(lab, N_PER_CLASS)
            parts_all.append(df_lab)

            out_lab = os.path.join(
                PER_CLASS_DIR,
                f"synthetic_{safe_filename(lab)}_{N_PER_CLASS}.csv"
            )
            df_lab.to_csv(out_lab, index=False)
            print(f"Saved {lab}: {out_lab}")

        # 3) Combined dataset (shuffled)
        synth_all = (
            pd.concat(parts_all, axis=0)
            .sample(frac=1, random_state=RANDOM_SEED)
            .reset_index(drop=True)
        )

        out_all = os.path.join(
            OUT_DIR,
            f"synthetic_ctgan_combined.csv"
        )
        synth_all.to_csv(out_all, index=False)
        print("Saved combined:", out_all, synth_all.shape)
        
    else:
        n_benign = TOTAL_SYN_BALANCED // 2
        n_mal_total = TOTAL_SYN_BALANCED - n_benign
        mal_labels = [c for c in classes if c != str(BENIGN_NAME)]
        n_per_mal = max(n_mal_total // max(1, len(mal_labels)), 1)
        parts = []
        if str(BENIGN_NAME) in classes:
            parts.append(sample_exact_class_ctgan(str(BENIGN_NAME), n_benign))
        for lab in mal_labels:
            parts.append(sample_exact_class_ctgan(str(lab), n_per_mal))
        synth_bal = pd.concat(parts, axis=0).sample(frac=1, random_state=RANDOM_SEED).reset_index(drop=True)
        out = os.path.join(OUT_DIR, "synthetic_ctgan_conditional_balanced.csv")
        synth_bal.to_csv(out, index=False)
        print("Wrote:", out, synth_bal.shape)


## 6) Post-Generation Sanity Check

In [ ]:
def _numeric_cols(df, exclude=None):
    exclude = set(exclude or [])
    cols = []
    for c in df.columns:
        if c in exclude:
            continue
        if pd.api.types.is_numeric_dtype(df[c]):
            cols.append(c)
    return cols

def _binary_oh_cols(df, exclude=None):
    exclude = set(exclude or [])
    cols = []
    for c in df.columns:
        if c in exclude:
            continue
        if pd.api.types.is_numeric_dtype(df[c]):
            u = set(pd.to_numeric(df[c], errors="coerce").dropna().unique().tolist())
            if u.issubset({0,1,0.0,1.0}):
                cols.append(c)
    return cols

def sanity_check_generated_ctgan_from_exports(return_data=True):
    #Validates synthetic outputs.

    syn_combined = None
    syn_by_class = {}
    syn_u = None

    if TRAIN_MODE in ("conditional", "both"):  
        assert TRAIN_MODE in ("conditional", "both"), "This sanity check is for conditional generation."
        assert "Xtr_cond" in globals(), "Xtr_cond not found (need conditional training dataframe)."

        ref_cols = list(Xtr_cond.columns)
        real_classes = sorted(Xtr_cond[LABEL_COL].dropna().astype(str).unique().tolist())
        mal_labels = [c for c in real_classes if c != str(BENIGN_NAME)]
        assert len(mal_labels) > 0, "No malicious classes found in Xtr_cond."

        expected_benign = len(mal_labels) * N_PER_CLASS
        expected_per_mal = N_PER_CLASS

        bin_cols_local = _binary_oh_cols(Xtr_cond, exclude=[LABEL_COL])

        def _check_one_df(df_syn: pd.DataFrame, context: str):
            # Schema
            assert list(df_syn.columns) == ref_cols, f"[{context}] Schema mismatch (columns differ)"

            # One-hot sanity: remain binary
            for c in bin_cols_local:
                u = set(pd.to_numeric(df_syn[c], errors="coerce").dropna().unique().tolist())
                assert u.issubset({0, 1, 0.0, 1.0}), f"[{context}] One-hot '{c}' has non-binary values (sample={sorted(list(u))[:10]})"

        syn_combined = None
        syn_by_class = {}

        if GENERATE_PER_CLASS:
            per_dir = os.path.join(OUT_DIR, "synthetic_per_class")
            assert os.path.isdir(per_dir), f"Missing per-class directory: {per_dir}"

            files = sorted([f for f in os.listdir(per_dir) if f.lower().endswith(".csv")])
            assert len(files) > 0, f"No per-class CSVs found in: {per_dir}"

            found_labels = set()

            for fn in files:
                path = os.path.join(per_dir, fn)
                df_syn = pd.read_csv(path, engine="python", on_bad_lines="skip")

                # Must be single-label file
                labs = df_syn[LABEL_COL].astype(str).unique().tolist()
                assert len(labs) == 1, f"[{fn}] contains multiple labels: {labs[:5]}"
                lab = str(labs[0])
                found_labels.add(lab)

                # Expected row counts
                if lab == str(BENIGN_NAME):
                    assert len(df_syn) == expected_benign, f"[{fn}] Benign rows={len(df_syn)} != expected {expected_benign}"
                else:
                    assert len(df_syn) == expected_per_mal, f"[{fn}] {lab} rows={len(df_syn)} != expected {expected_per_mal}"

                _check_one_df(df_syn, context=f"per-class:{fn}")
                syn_by_class[lab] = df_syn

            # Ensure all real classes exist as per-class exports
            missing = sorted(set(real_classes) - found_labels)
            assert not missing, f"Missing per-class exports for classes: {missing}"

            print(f"Per-class files OK: {len(files)} files in {per_dir}")

            # Combined file
            combined_path = os.path.join(OUT_DIR, "synthetic_ctgan_combined.csv")
            assert os.path.exists(combined_path), f"Missing combined file: {combined_path}"
            syn_combined = pd.read_csv(combined_path, engine="python", on_bad_lines="skip")

            _check_one_df(syn_combined, context="combined")
            # Combined must contain all classes
            syn_classes = sorted(syn_combined[LABEL_COL].dropna().astype(str).unique().tolist())
            assert set(real_classes).issubset(set(syn_classes)), "Combined output missing some classes."

            # Combined must be 50/50: benign == sum(malicious)
            vc = syn_combined[LABEL_COL].astype(str).value_counts()
            benign_n = int(vc.get(str(BENIGN_NAME), 0))
            mal_n = int(vc.drop(str(BENIGN_NAME), errors="ignore").sum())
            assert benign_n == mal_n, f"Combined not 50/50: benign={benign_n}, mal_total={mal_n}"

            # And must match expected totals
            assert benign_n == expected_benign, f"Combined benign rows={benign_n} != expected {expected_benign}"
            assert mal_n == expected_benign, f"Combined malicious total={mal_n} != expected {expected_benign}"

            print("Combined file OK:", combined_path, "shape:", syn_combined.shape)

            # Numeric drift report (combined)
            num_cols_local = _numeric_cols(Xtr_cond, exclude=bin_cols_local + [LABEL_COL])
            if num_cols_local:
                real_mu = Xtr_cond[num_cols_local].mean()
                syn_mu  = syn_combined[num_cols_local].mean()
                delta = (syn_mu - real_mu).abs().sort_values(ascending=False)
                print("Top mean shifts (abs) [combined]:")
                display(delta.head(10))

            print("Validation OK (GENERATE_PER_CLASS=True)")

        else:
            # Balanced single-file mode
            syn_path = os.path.join(OUT_DIR, "synthetic_ctgan_conditional_balanced.csv")
            assert os.path.exists(syn_path), f"Missing synthetic file: {syn_path}"
            syn_combined = pd.read_csv(syn_path, engine="python", on_bad_lines="skip")

            _check_one_df(syn_combined, context="balanced-single")
            syn_classes = sorted(syn_combined[LABEL_COL].dropna().astype(str).unique().tolist())
            assert set(real_classes).issubset(set(syn_classes)), "Balanced synthetic missing some classes."

            print("Balanced single synthetic OK:", syn_path, "shape:", syn_combined.shape)
            print("Validation OK (GENERATE_PER_CLASS=False)")

    else:
        # Unconditional
        syn_u = None
        if TRAIN_MODE in ("unconditional", "both"):
            syn_u_path = os.path.join(OUT_DIR, "synthetic_ctgan_unconditional.csv")
            if os.path.exists(syn_u_path):
                syn_u = pd.read_csv(syn_u_path, engine="python", on_bad_lines="skip")
                assert list(syn_u.columns) == list(Xtr_uncond.columns), "Schema mismatch: synthetic unconditional != real unconditional"
                print("Unconditional synthetic OK:", syn_u_path, syn_u.shape)
                
    if return_data:
        return syn_combined, syn_by_class, syn_u
    return None

syn_combined, syn_by_class, syn_u = sanity_check_generated_ctgan_from_exports(return_data=True)




## 7) Figure Page (Plots)
### Real vs Synthetic - Saves a PNG

In [ ]:
MODEL_NAME = "CTGAN"
FIG_OUT = os.path.join(OUT_DIR, f"figure_real_vs_synthetic_{MODEL_NAME}.png")

# helpers
def binary_cols(df, exclude=None):
    exclude = set(exclude or [])
    cols = []
    for c in df.columns:
        if c in exclude:
            continue
        if pd.api.types.is_numeric_dtype(df[c]):
            u = set(pd.to_numeric(df[c], errors="coerce").dropna().unique())
            if u.issubset({0, 1, 0.0, 1.0}):
                cols.append(c)
    return cols

def numeric_cols(df, exclude=None):
    exclude = set(exclude or [])
    return [c for c in df.columns if c not in exclude and pd.api.types.is_numeric_dtype(df[c])]

def pick_key_numeric(df, num_cols):
    preferred = ["Dur","Load","SrcLoad","DstLoad","SrcBytes","DstBytes",
                 "SrcPkts","DstPkts","sMeanPktSz","Loss"]
    picked = [c for c in preferred if c in num_cols]
    if len(picked) >= 6:
        return picked[:6]
    return list(df[num_cols].var().sort_values(ascending=False).head(6).index)

# prep
real = Xtr_cond.copy()
syn  = syn_combined.copy() if syn_combined is not None else syn_u.copy()

has_real_label = (LABEL_COL in real.columns)
has_syn_label  = (LABEL_COL in syn.columns)

# features: use only columns that exist in BOTH
feat_cols_real = [c for c in real.columns if c != LABEL_COL]
common_feat_cols = [c for c in feat_cols_real if c in syn.columns]

missing_in_syn = sorted(set(feat_cols_real) - set(common_feat_cols))
if missing_in_syn:
    print(f"Warning: syn is missing {len(missing_in_syn)} feature cols. Example:", missing_in_syn[:10])

R = real[common_feat_cols].apply(pd.to_numeric, errors="coerce").fillna(0)
S = syn[common_feat_cols].apply(pd.to_numeric, errors="coerce").fillna(0)

# choose bin/num cols based on REAL, but restricted to common cols
bin_cols_all = binary_cols(real, exclude=[LABEL_COL])
bin_cols = [c for c in bin_cols_all if c in common_feat_cols]
num_cols = [c for c in numeric_cols(real, exclude=bin_cols_all + ([LABEL_COL] if has_real_label else []))
            if c in common_feat_cols]

key_num = pick_key_numeric(real, num_cols) if len(num_cols) else []
if not key_num:
    print("Warning: No numeric columns found for panel (B)/(C).")

sample_n = 20000
R_s = R.sample(n=min(sample_n, len(R)), random_state=RANDOM_SEED)
S_s = S.sample(n=min(sample_n, len(S)), random_state=RANDOM_SEED)

# figure
fig = plt.figure(figsize=(16, 16))

# (A) Label distribution (only if both have labels)
axA = fig.add_subplot(3, 2, 1)
if has_real_label and has_syn_label:
    rc = real[LABEL_COL].value_counts()
    sc = syn[LABEL_COL].value_counts()
    idx = rc.index.union(sc.index)
    x = np.arange(len(idx))
    axA.bar(x - 0.2, rc.reindex(idx, fill_value=0), 0.4, label="Real")
    axA.bar(x + 0.2, sc.reindex(idx, fill_value=0), 0.4, label="Synthetic")
    axA.set_xticks(x)
    axA.set_xticklabels(idx, rotation=45, ha="right")
    axA.set_title("(A) Attack Type distribution")
    axA.legend()
else:
    axA.axis("off")
    axA.text(
        0.5, 0.5,
        f"(A) Skipped label plot\n'{LABEL_COL}' missing in "
        f"{'real' if not has_real_label else ''}{' & ' if (not has_real_label and not has_syn_label) else ''}"
        f"{'synthetic' if not has_syn_label else ''}",
        ha="center", va="center"
    )

# (B) Numeric distributions
axB = fig.add_subplot(3, 2, 2)
axB.axis("off")
axB.set_title("(B) Numeric feature distributions")
if key_num:
    for i, c in enumerate(key_num[:6]):
        ax = fig.add_axes([0.55 + (i % 3) * 0.13, 0.70 - (i // 3) * 0.18, 0.12, 0.15])
        ax.hist(real[c].dropna(), bins=50, alpha=0.5, density=True, label="Real" if i == 0 else None)
        ax.hist(syn[c].dropna() if c in syn.columns else [], bins=50, alpha=0.5, density=True,
                label="Synthetic" if i == 0 else None)
        ax.set_title(c, fontsize=9)
        ax.tick_params(labelsize=7)

# (C) Correlation heatmaps
axC1 = fig.add_subplot(3, 2, 3)
axC2 = fig.add_subplot(3, 2, 4)
if key_num:
    corr_r = real[key_num].corr()
    corr_s = syn[key_num].corr() if all(c in syn.columns for c in key_num) else pd.DataFrame()
    axC1.imshow(corr_r); axC1.set_title("(C1) Real correlations")
    axC2.imshow(corr_s if not corr_s.empty else np.zeros((len(key_num), len(key_num))))
    axC2.set_title("(C2) Synthetic correlations")
else:
    axC1.axis("off"); axC2.axis("off")
    axC1.text(0.5, 0.5, "(C) Skipped — no numeric cols", ha="center", va="center")

# (D) PCA projection
axD = fig.add_subplot(3, 2, 5)
pca = PCA(n_components=2, random_state=RANDOM_SEED)
Z = pca.fit_transform(pd.concat([R_s, S_s], axis=0).values)
axD.scatter(Z[:len(R_s), 0], Z[:len(R_s), 1], s=4, alpha=0.4, label="Real")
axD.scatter(Z[len(R_s):, 0], Z[len(R_s):, 1], s=4, alpha=0.4, label="Synthetic")
axD.set_title("(D) PCA projection")
axD.legend()

# (E) One-hot activation rates
axE = fig.add_subplot(3, 2, 6)
if bin_cols:
    diff = (syn[bin_cols].mean() - real[bin_cols].mean()).abs().sort_values(ascending=False).head(20)
    axE.bar(diff.index, real[diff.index].mean(), alpha=0.6, label="Real")
    axE.bar(diff.index, syn[diff.index].mean(), alpha=0.6, label="Synthetic")
    axE.set_title("(E) One-hot activation rates")
    axE.tick_params(axis="x", labelrotation=45)
    axE.legend()
else:
    axE.axis("off")
    axE.text(0.5, 0.5, "(E) Skipped — no binary one-hot cols", ha="center", va="center")

plt.suptitle(f"Real vs Synthetic Comparison ({MODEL_NAME})", fontsize=16)
plt.tight_layout()
plt.savefig(FIG_OUT, dpi=300, bbox_inches="tight")
plt.show()

print("Saved figure to:", FIG_OUT)


## 8) Summary

In [ ]:
summary = {
    # Model and Setup
    "model": "CTGAN",
    "library": "ctgan",
    "library_version": pkg_resources.get_distribution("ctgan").version,
    "python_version": platform.python_version(),
    "torch_version": torch.__version__,

    # Training Mode
    "train_mode": TRAIN_MODE,                
    "label_col": LABEL_COL if TRAIN_MODE != "unconditional" else None,
    "generate_per_class": bool(GENERATE_PER_CLASS),
    "n_per_class": int(N_PER_CLASS) if GENERATE_PER_CLASS else None,
    "uncond_samples": int(UNCOND_SAMPLES) if TRAIN_MODE != "conditional" else None,

    # Dataset
    "train_rows_conditional": int(Xtr_cond.shape[0]) if TRAIN_MODE != "unconditional" else None,
    "train_rows_unconditional": int(Xtr_uncond.shape[0]) if TRAIN_MODE != "conditional" else None,
    "num_features": int(Xtr_cond.shape[1] - 1) if TRAIN_MODE != "unconditional" else int(Xtr_uncond.shape[1]),
    "num_classes": int(Xtr_cond[LABEL_COL].nunique()) if TRAIN_MODE != "unconditional" else None,

    # Hyperparameters 
    "epochs": int(EPOCHS),
    "batch_size": int(BATCH),

    # Runtime 
    "device": DEVICE,
    "cuda_available": bool(torch.cuda.is_available()),
    "training_time_seconds": round(train_duration_sec, 2),
    "training_time_minutes": round(train_duration_sec / 60, 2),
    "samples_per_second": round(
        (N_PER_CLASS * Xtr_cond[LABEL_COL].nunique()) / train_duration_sec, 2
    ) if GENERATE_PER_CLASS and TRAIN_MODE != "unconditional" else None,

    # System
    "hostname": platform.node(),
    "timestamp": time.strftime("%Y-%m-%d %H:%M:%S"),
}

out_json = os.path.join(OUT_DIR, "augmentation_ctgan_training_summary.json")
with open(out_json, "w") as f:
    json.dump(summary, f, indent=2)

print("Wrote:", out_json)
summary
